# Mini Data Quality Audit
## Dataset: week2_customer_transactions_messy.csv

In [11]:
import pandas as pd, numpy as np
df=pd.read_csv('week2_customer_transactions_messy.csv')
df

,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
0,T0001,C100,2026-01-05,120.50,EUR,card,completed,DE,2026-01-05
1,T0002,C101,2026/01/06,0.00,EUR,CARD,completed,de,2026-01-20
2,T0003,C102,06-01-2026,-35.00,USD,bank_transfer,completed,US,2026-01-07
3,T0004,NaN,2026-01-07,250.00,EUR,card,pending,FR,2026-01-08
4,T0005,C104,2026-01-07,89.99,EURO,cash,completed,DE,2026-01-09
5,T0006,C105,2026-01-08,19.99,EUR,card,cancelled,DE,2026-04-15
6,T0006,C105,2026-01-08,19.99,EUR,card,cancelled,DE,2026-04-15
7,T0007,C106,2026-02-30,47.00,EUR,card,completed,DE,2026-02-15
8,T0008,C107,2026-01-10,1000000.00,EUR,card,completed,DE,2026-01-10
9,T0009,C108,2026-01-11,NaN,EUR,bank_transfer,completed,NL,NaN


## 1. Dataset Description & Business Use Case
The dataset contains customer transaction records with IDs, dates, amounts, currencies, payment methods, statuses, regions, and update timestamps.

**Potential business uses**
- Revenue and sales reporting  
- Customer behavior analysis  
- Fraud / anomaly detection  
- Payment channel performance tracking  
- Regional performance dashboards


## 2. Data Quality Issues by Dimension
Completeness Issues : Some records contain missing values in important columns such as customer ID, amount, payment method, and last updated date.

Uniqueness Issues : Some transaction IDs are duplicated. This can lead to double counting of revenue.

Validity Issues : Some values are incorrect, such as Negative transaction amount , Zero amount, Invalid date values, Missing numeric values, Consistency Issues

Some values use different formats: card / CARD, DE / de, EUR / EURO, Different date formats, Integrity Issues

One transaction amount is extremely high (1,000,000), which may be an outlier or entry error.


In [12]:

amount=pd.to_numeric(df['amount'],errors='coerce')
date_ok=pd.to_datetime(df['transaction_date'],errors='coerce',dayfirst=True).notna()

kpis=pd.DataFrame({
'KPI':['Completeness Rate','Duplication Rate','Error Rate','Missing Value Count'],
'Value':[
round(1-df.isna().sum().sum()/(df.shape[0]*df.shape[1]),4),
round(df.duplicated(subset=['transaction_id']).mean(),4),
round((~date_ok | amount.isna() | (amount<=0)).mean(),4),
int(df.isna().sum().sum())
]})
kpis


,KPI,Value
0,Completeness Rate,0.9596
1,Duplication Rate,0.0909
2,Error Rate,0.3636
3,Missing Value Count,4.0000


## KPI Interpretation
- **Completeness Rate (~95.96%)**: Generally good, but missing critical fields still impact analysis.  
- **Duplication Rate (~9.09%)**: High enough to distort revenue and transaction counts.  
- **Error Rate (~36.36%)**: Significant quality issues needing urgent remediation.  


In [13]:
kpis={}
kpis['Completeness Rate']=1-(df.isna().sum().sum()/(df.shape[0]*df.shape[1]))
kpis['Duplication Rate']=df.duplicated(subset=['transaction_id']).mean()
amount=pd.to_numeric(df['amount'], errors='coerce')
date_ok=pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed').notna()
kpis['Error Rate']=(amount.isna() | (amount<0) | ~date_ok).mean()
kpi_df = pd.DataFrame({'KPI':list(kpis.keys()), 'Value':list(kpis.values())})
kpi_df['Value (%)'] = (kpi_df['Value']*100).round(1)
kpi_df

,KPI,Value,Value (%)
0,Completeness Rate,0.959596,96.0
1,Duplication Rate,0.090909,9.1
2,Error Rate,0.272727,27.3


## 4. Audit Summary
Rule 1 : Transaction ID must not be empty and must be unique.

Rule 2 : Transaction amount must be numeric and greater than zero.

Rule 3 : Transaction date must be a valid date.

Rule 4 : Payment method must only contain approved values Card, Cash, Bank Transfer.

Rule 5 : Currency codes should follow standard values such as EUR, USD, GBP.


## 5. Suggested Cleaning Steps
1. Standardize date formats to YYYY-MM-DD.  
2. Convert amount to numeric and flag non-positive values.  
3. Remove or merge duplicate transaction IDs.  
4. Fill or investigate missing customer/payment fields.  
5. Normalize categories (card/CARD, DE/de, EUR/EURO).  
6. Review suspicious outliers with business owners.  
7. Enforce automated validation rules in ETL pipeline.
